## Select residues within certain distance from the substrate in pymol

In [1]:
''' find residues within certain distance from the substrate'''
select near_residues_15A, byres (protein and name CA within 15 of (substrate and name P))

# select protein and ligand and name them as protein and substrate 
# protein and name CA: selects only the alpha-carbon atoms from the protein
# substrate and name P: selects only the phosphate atoms from the substrate (nucleic acids)
# within 18 of Finds atom within 18 A of each other 
# byres: ensures that entire resiudes are selected if any of their atoms meet the criteria 

SyntaxError: invalid syntax (4136223627.py, line 2)

In [2]:
''' save residues position indices in a text file '''
# Step 1: Initialize an empty list
residue_indices_15A = []

# Step 2: Collect residue indices from the selection
cmd.iterate("near_residues_15A", "residue_indices_20A.append(resi)")

# Step 3: Remove duplicates and sort the list
residue_indices_15A = sorted(set(residue_indices_15A), key=lambda x: int(x))

# Step 4: Define the file path
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_AF3_15A_residue_indices.txt'

# Step 5: Save the list to a text file
with open(file_path, 'w') as outfile: outfile.write(', '.join(residue_indices_15A))

NameError: name 'cmd' is not defined

## Convert the active site residue indices in the substrate-bound homolog to those in target protein

In [3]:
from Bio import AlignIO

In [4]:
'''import the residues text file generated from pymol'''
# Step 1: Define the file path
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/18A_residue_indices.txt'

# Step 2: Read the contents of the file
with open(file_path, 'r') as infile:
    data = infile.read()

# Step 3: Convert the string data to a list
residue_indices = data.split(', ')

# Step 4: (Optional) Convert residue indices to integers
residue_indices = [int(res) for res in residue_indices]

In [5]:
# MMLV_deltaRNaseH_substrate_bound has some wired index annotations
residue_indices_adjusted =[]
for r in residue_indices:
    if r <449:
        r2=r-23
    if r>449:
        r2 = r-23-6
    residue_indices_adjusted.append(r2)

In [6]:
'''if use a homolog structure, the codes below can convert the homolog residue positions to your target protein positions'''
alignment = AlignIO.read('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/Alignment_MMLV_deltaRNaseH_PE6d.fasta', "fasta")

# define the function to map the homolog residues to the target protein residues by sequence alignment
def index_convert(resList):

  outList = []
  outList_num = []
  res_counter_A = 0
  res_counter_B = 0
  for pos in range(alignment.get_alignment_length()):

    if alignment[1][pos] != '-':
      res_counter_B += 1

    if alignment[0][pos] != '-':
      res_counter_A += 1
      if res_counter_A in resList:
        outList.append(alignment[1][res_counter_B])
        outList_num.append(res_counter_B)
    
  return(outList,outList_num)

In [7]:
residues, num = index_convert(residue_indices_adjusted)
PDB_fix_Res = num
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PDB_fix_Res.txt'
with open(file_path, 'w') as outfile:
    for item in PDB_fix_Res:
        outfile.write(str(item) + '\n')

## Compare fixed residues between AF3 and experimental homolog structure 

In [1]:
import re

# function to read the text file 
def read_file(file_path):
    with open(file_path, 'r') as infile:
        data = infile.read()
        # Split on commas and newlines, strip whitespace, and filter out empty strings
        data_list = [item.strip() for item in re.split(r',\s*|\n', data) if item.strip()]
        # Convert the list items to integers
        data_list = [int(item) for item in data_list]
    return data_list

# read the text file
PE6c_AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_AF3_15A_residue_indices.txt')
PE6c_substrate_bound_AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_substrate_bound_AF3_15A_residue_indices.txt')
PE6c_AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_AF3_18A_residue_indices.txt')
PE6c_substrate_bound_AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_substrate_bound_AF3_18A_residue_indices.txt')
PE6c_AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_AF3_20A_residue_indices.txt')
PE6c_substrate_bound_AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_substrate_bound_AF3_20A_residue_indices.txt')

In [8]:
# Convert the lists to sets for efficient comparison
PE6c_AF3_15A_set = set(PE6c_AF3_15A)
PE6c_substrate_bound_AF3_15A_set = set(PE6c_substrate_bound_AF3_15A)
PE6c_AF3_18A_set = set(PE6c_AF3_18A)
PE6c_substrate_bound_AF3_18A_set = set(PE6c_substrate_bound_AF3_18A)
PE6c_AF3_20A_set = set(PE6c_AF3_20A)
PE6c_substrate_bound_AF3_20A_set = set(PE6c_substrate_bound_AF3_20A)

def compare_indices(set1, set2, set1_name='Set 1', set2_name='Set 2'):
    """
    Compares two sets of indices and prints the common and unique indices.

    Parameters:
    - set1: The first set of indices.
    - set2: The second set of indices.
    - set1_name: A string representing the name of the first set (for display purposes).
    - set2_name: A string representing the name of the second set (for display purposes).

    Returns:
    A dictionary containing:
    - 'common_indices': Indices present in both sets.
    - 'unique_to_set1': Indices unique to the first set.
    - 'unique_to_set2': Indices unique to the second set.
    """
    # Find indices that appear in both sets (common indices)
    common_indices = set1 & set2  # Intersection of the two sets

    # Find indices unique to each set
    unique_to_set1 = set1 - set2   # Indices in set1 but not in set2
    unique_to_set2 = set2 - set1   # Indices in set2 but not in set1

    # Output the results
    print(f"Indices present in both {set1_name} and {set2_name}:")
    print(sorted(common_indices))

    print(f"\nIndices unique to {set1_name}:")
    print(sorted(unique_to_set1))

    print(f"\nIndices unique to {set2_name}:")
    print(sorted(unique_to_set2))

    # Return the results in a dictionary
    return {
        'common_indices': common_indices,
        'unique_to_set1': unique_to_set1,
        'unique_to_set2': unique_to_set2
    }

In [9]:
compare_termination_initiation = compare_indices(PE6c_AF3_15A_set, PE6c_substrate_bound_AF3_15A_set, 'PE6c_AF3_15A', 'PE6c_substrate_bound_AF3_15A')

Indices present in both PE6c_AF3_15A and PE6c_substrate_bound_AF3_15A:
[53, 55, 99, 100, 101, 102, 103, 104, 105, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 149, 150, 151, 152, 153, 154, 155, 169, 171, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 211, 212, 213, 214, 215, 216, 217, 218, 249, 250, 251, 257, 258, 259, 260, 261, 262, 263, 264, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 305, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 364, 365, 366, 367, 368, 369, 370, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 402, 406, 407, 409, 410, 411, 412, 413, 414, 415, 416, 417, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456]

Indices unique to PE6c_AF3_15A:
[52, 54, 56, 

In [11]:
compare_termination_elongation = compare_indices(PE6c_AF3_18A_set, PE6c_substrate_bound_AF3_18A_set, 'PE6c_AF3_18A', 'PE6c_substrate_bound_AF3_18A')

Indices present in both PE6c_AF3_18A and PE6c_substrate_bound_AF3_18A:
[51, 52, 53, 54, 55, 56, 62, 63, 64, 65, 66, 67, 68, 69, 73, 77, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 141, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 168, 169, 170, 171, 172, 173, 180, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 245, 246, 247, 248, 249, 250, 251, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 331, 334, 335, 338, 339, 342, 361, 362, 363, 364, 365, 366, 3

In [15]:
compare_termination_AF3 = compare_indices(PE6c_AF3_20A_set, PE6c_substrate_bound_AF3_20A_set, 'PE6c_AF3_20A', 'PE6c_substrate_bound_AF3_20A')


Indices present in both PE6c_AF3_20A and PE6c_substrate_bound_AF3_20A:
[49, 50, 51, 52, 53, 54, 55, 56, 57, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 73, 74, 76, 77, 80, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 161, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 178, 179, 180, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 202, 206, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 244, 245, 246, 247, 248, 249, 250, 251, 252, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 3

## Generate final fixed residues by distance from combination of AF3 substrate bound structure and AF3 structure 

In [2]:
import re

# function to read the text file 
def read_file(file_path):
    with open(file_path, 'r') as infile:
        data = infile.read()
        # Split on commas and newlines, strip whitespace, and filter out empty strings
        data_list = [item.strip() for item in re.split(r',\s*|\n', data) if item.strip()]
        # Convert the list items to integers
        data_list = [int(item) for item in data_list]
    return data_list

# read the text file
PE6c_AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_AF3_15A_residue_indices.txt')
PE6c_substrate_bound_AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_substrate_bound_AF3_15A_residue_indices.txt')
PE6c_AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_AF3_18A_residue_indices.txt')
PE6c_substrate_bound_AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_substrate_bound_AF3_18A_residue_indices.txt')
PE6c_AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_AF3_20A_residue_indices.txt')
PE6c_substrate_bound_AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_substrate_bound_AF3_20A_residue_indices.txt')

In [3]:
# Convert the lists to sets for efficient comparison
PE6c_AF3_15A_set = set(PE6c_AF3_15A)
PE6c_substrate_bound_AF3_15A_set = set(PE6c_substrate_bound_AF3_15A)
PE6c_AF3_18A_set = set(PE6c_AF3_18A)
PE6c_substrate_bound_AF3_18A_set = set(PE6c_substrate_bound_AF3_18A)
PE6c_AF3_20A_set = set(PE6c_AF3_20A)
PE6c_substrate_bound_AF3_20A_set = set(PE6c_substrate_bound_AF3_20A)


In [6]:
def fixed_residues_distance(AF3, AF3_substrate_bound, output_file):
    
    fixed_residues = AF3_substrate_bound 

    # Find indices unique to each set
    unique_to_AF3= AF3 - AF3_substrate_bound   # Indices in AF3 but not in experiment

    for index in unique_to_AF3:
            fixed_residues.add(index)

    with open(output_file, 'w') as file:
        for residue in sorted(fixed_residues):  # Sort for readability
            file.write(f"{residue}\n")

    return fixed_residues

In [20]:
PE6c_fixed_residues_15A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_fixed_residues_15A.txt'
PE6d_fixed_residues_15A = fixed_residues_distance(PE6c_AF3_15A_set, PE6c_substrate_bound_AF3_15A_set, PE6c_fixed_residues_15A_output_file)
PE6d_fixed_residues_15A

{52,
 53,
 54,
 55,
 56,
 62,
 63,
 65,
 67,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 169,
 171,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 246,
 249,
 250,
 251,
 257,
 258,
 259,
 260,
 261,
 262,
 263,
 264,
 269,
 270,
 271,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 280,
 281,
 282,
 283,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 313,
 314,
 315,
 316,
 317,
 318,
 319,
 320,
 321,
 322,
 323,
 324,
 325,
 363,
 364,
 365,
 366,
 367,
 368,
 369,
 370,
 385,
 386,
 387,
 388,
 389,
 390,
 391,
 392,
 393,
 394,
 395,
 396,
 397,
 39

In [21]:
PE6c_fixed_residues_18A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_fixed_residues_18A.txt'
PE6d_fixed_residues_18A = fixed_residues_distance(PE6c_AF3_18A_set, PE6c_substrate_bound_AF3_18A_set, PE6c_fixed_residues_18A_output_file)
PE6d_fixed_residues_18A

{51,
 52,
 53,
 54,
 55,
 56,
 57,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 73,
 76,
 77,
 80,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 168,
 169,
 170,
 171,
 172,
 173,
 179,
 180,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 220,
 245,
 246,
 247,
 248,
 249,
 250,
 251,
 252,
 254,
 256,
 257,
 258,
 259,
 260,
 261,
 262,
 263,
 264,
 265,
 268,
 269,
 270,
 271,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 280,
 281,
 282,
 283,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307

In [7]:
PE6c_fixed_residues_20A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6c/PE6c_fixed_residues_20A.txt'
PE6c_fixed_residues_20A = fixed_residues_distance(PE6c_AF3_20A_set, PE6c_substrate_bound_AF3_20A_set, PE6c_fixed_residues_20A_output_file)
PE6c_fixed_residues_20A


{21,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 202,
 206,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 220,
 243,
 244,
 245,
 246,
 247,
 248,
 249,
 250,
 251,
 252,
 253,
 254,
 255,
 256,
 257,
 258,
 259,
 260,
 261,
 262,
 263,
 264,
 265,
 266,
 268,
 269,
 270,
 271,
 272,
 273,
 27

In [12]:
list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 24, 27, 30, 32, 41, 42, 43, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 85, 88, 90, 91, 93, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 176, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 202, 206, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 226, 227, 230, 231, 234, 235, 236, 238, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 350, 351, 352, 355, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 378, 380, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 425, 426, 429, 430, 431, 432, 434, 435, 438, 439, 440, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 464, 466, 467, 468, 469, 470, 472, 475, 476, 477, 478, 479, 480, 483, 497]
len(list)


394

318

323